Preparations:
- load prediction.csv
    - drop the lines with the non-used ifs
    - if there is more then one if, take mean

- add column "sample" to master table
- join them
- drop unnecessary columns (matadata, paths, ranks)

In [84]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import ast
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, f1_score, precision_recall_curve, confusion_matrix

# Configuration
CSV_PATH =  "/home/bri/pymol/data/predictions/preds_snir.csv" 
MASTER_PATH =  "/home/bri/pymol/data/3ab/table/master_table_snir.csv" 
YAML_PATH = "/home/bri/pymol/EDA/static/metric_directions.yaml"
OUTPUT_DIR = Path("/home/bri/pymol/EDA/static/snir_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


KEEP_INTERFACES = {"A,C", "B,C"}   # as option if there is more than one, default A,B
palette = {"binder": "#2196F3", "ala_scan": "#FF5722"}


# Load data
preds_df = pd.read_csv(CSV_PATH)

preds_df.head()
preds_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 92 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   sample                                  105 non-null    str    
 1   binding                                 105 non-null    bool   
 2   interface                               105 non-null    str    
 3   af3_pae                                 105 non-null    float64
 4   af3_plddt                               105 non-null    float64
 5   af3_interface_dG                        105 non-null    float64
 6   af3_interface_dG_SASA_ratio             105 non-null    float64
 7   af3_ipae                                105 non-null    float64
 8   af3_iplddt                              105 non-null    float64
 9   af3_ipsae                               105 non-null    float64
 10  af3_pdockq                              105 non-null    float64
 11  af3_

In [85]:
df_if = preds_df[preds_df["interface"].str.strip().isin(KEEP_INTERFACES)].copy()
print(f"Loaded {len(df_if)} rows. Interfaces: {df_if['interface'].unique()}")

df_if.head(10)


Loaded 70 rows. Interfaces: <StringArray>
['A,C', 'B,C']
Length: 2, dtype: str


,sample,binding,interface,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,...,boltz_template_plddt,boltz_template_iplddt,boltz_template_pde,boltz_template_ipde,boltz_template_per_chain_pair_iptm,boltz_template_model_path,boltz_template_ipsae,boltz_template_pdockq,boltz_template_pdockq2,boltz_template_lis
1,5ig7_abc_cut,False,"A,C",2.572969,95.599717,10.97,1.38,2.321875,94.630252,0.780104,...,0.987334,0.987299,0.258769,0.284754,0.948463,/scicore/home/schwede/barta0000/scoring_pipeli...,0.921477,0.2352,0.9699,0.8588
2,5ig7_abc_cut,False,"B,C",2.572969,95.599717,5.13,0.67,1.720000,95.189670,0.790385,...,0.987334,0.987299,0.258769,0.284754,0.956973,/scicore/home/schwede/barta0000/scoring_pipeli...,0.928806,0.2332,0.9688,0.8586
4,5ig7_abc_cut_A_Y66A,False,"A,C",2.572738,95.767986,17.48,2.33,2.219231,94.959764,0.779923,...,0.986631,0.985549,0.257634,0.291466,0.935394,/scicore/home/schwede/barta0000/scoring_pipeli...,0.898429,0.2315,0.9617,0.8487
5,5ig7_abc_cut_A_Y66A,False,"B,C",2.572738,95.767986,0.73,0.10,1.700000,95.167570,0.792646,...,0.986631,0.985549,0.257634,0.291466,0.945723,/scicore/home/schwede/barta0000/scoring_pipeli...,0.907364,0.2149,0.9585,0.8409
7,5ifj_abc_cut_A_S64A,False,"A,C",2.539446,95.613237,23.78,2.92,2.434375,94.266763,0.775252,...,0.984658,0.983850,0.258604,0.295100,0.932908,/scicore/home/schwede/barta0000/scoring_pipeli...,0.892828,0.2290,0.9581,0.8494
8,5ifj_abc_cut_A_S64A,False,"B,C",2.539446,95.613237,5.41,0.70,1.710000,95.361092,0.782671,...,0.984658,0.983850,0.258604,0.295100,0.944936,/scicore/home/schwede/barta0000/scoring_pipeli...,0.904881,0.2280,0.9569,0.8415
10,5ifj_abc_cut_all_A,False,"A,C",3.597543,93.235998,52.47,5.49,2.602632,84.615603,0.216341,...,0.953720,0.922320,0.256769,0.294560,0.943772,/scicore/home/schwede/barta0000/scoring_pipeli...,0.912904,0.1090,0.6813,0.8530
11,5ifj_abc_cut_all_A,False,"B,C",3.597543,93.235998,16.68,4.56,1.806250,82.884986,0.162327,...,0.953720,0.922320,0.256769,0.294560,0.926717,/scicore/home/schwede/barta0000/scoring_pipeli...,0.872455,0.1163,0.6247,0.8453
13,5ig7_abc_cut_B_G114A,False,"A,C",2.436673,95.869087,24.31,3.01,2.137500,95.580074,0.803864,...,0.986487,0.986081,0.256536,0.278153,0.950070,/scicore/home/schwede/barta0000/scoring_pipeli...,0.924004,0.2335,0.9680,0.8583
14,5ig7_abc_cut_B_G114A,False,"B,C",2.436673,95.869087,0.09,0.01,1.631250,95.483511,0.816920,...,0.986487,0.986081,0.256536,0.278153,0.955974,/scicore/home/schwede/barta0000/scoring_pipeli...,0.927049,0.2016,0.9601,0.8512


In [93]:
num_cols = df_if.select_dtypes(include=['number']).columns
other_cols = [c for c in df_if.columns if c not in num_cols and c != 'sample']

agg_dict = {col: 'mean' for col in num_cols}
agg_dict.update({col: 'first' for col in other_cols})

df_1if = df_if.groupby('sample').agg(agg_dict).reset_index()

df_1if

,sample,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq,af3_pdockq2,...,binding,interface,chai_unconstrained_model_path,chai_constrained_model_path,chai_unconstrained_per_chain_ptm,chai_unconstrained_per_chain_pair_iptm,chai_constrained_per_chain_ptm,chai_constrained_per_chain_pair_iptm,boltz_free_model_path,boltz_template_model_path
0,5ifj_abc_cut,2.593883,95.404666,14.690,1.855,1.993750,95.036725,0.788184,0.20980,0.88085,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9473640322685242, 0.9504183530807495, 0.17...","[[[0.9473640322685242, 0.8909199237823486, 0.0...","[[0.9346736073493958, 0.941501259803772]]","[[[0.9346736073493958, 0.8659998178482056], [0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
1,5ifj_abc_cut_A_D111A,2.815847,95.282854,13.225,1.745,2.100000,94.730775,0.765295,0.18660,0.88195,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9472975134849548, 0.9492976665496826, 0.17...","[[[0.9472975134849548, 0.8904723525047302, 0.0...","[[0.934756338596344, 0.9413727521896362]]","[[[0.934756338596344, 0.8664464950561523], [0....",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
2,5ifj_abc_cut_A_K110A,2.790728,94.871984,19.975,2.845,2.275417,92.998114,0.769552,0.18030,0.85065,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9461949467658997, 0.9488812685012817, 0.16...","[[[0.9461949467658997, 0.8897757530212402, 0.0...","[[0.9350460767745972, 0.9415807127952576]]","[[[0.9350460767745972, 0.8660252094268799], [0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
3,5ifj_abc_cut_A_S64A,2.539446,95.613237,14.595,1.810,2.072188,94.813927,0.778962,0.19900,0.88450,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9445436000823975, 0.9495869874954224, 0.17...","[[[0.9445436000823975, 0.8924098014831543, 0.0...","[[0.9345250129699707, 0.9413044452667236]]","[[[0.9345250129699707, 0.8659787178039551], [0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
4,5ifj_abc_cut_A_W52A,3.005329,94.814333,46.435,5.940,2.303125,91.482646,0.695928,0.19125,0.78185,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9452440142631531, 0.9471760392189026, 0.16...","[[[0.9452440142631531, 0.8792759776115417, 0.0...","[[0.9345828890800476, 0.9411274194717407]]","[[[0.9345828890800476, 0.8663274645805359], [0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
5,5ifj_abc_cut_A_Y66A,2.701376,95.337838,17.485,2.305,2.249649,93.429048,0.763715,0.19335,0.84245,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9473956227302551, 0.9488828778266907, 0.16...","[[[0.9473956227302551, 0.8850024938583374, 0.0...","[[0.9346519708633423, 0.9413310289382935]]","[[[0.9346519708633423, 0.8664906620979309], [0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
6,5ifj_abc_cut_B_L116A,2.806607,95.032996,7.610,0.985,2.205952,93.484476,0.747441,0.17235,0.82990,...,False,"A,C",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9485109448432922, 0.9499985575675964, 0.17...","[[[0.9485109448432922, 0.8920019865036011, 0.0...","[[0.9348855018615723, 0.9416057467460632]]","[[[0.9348855018615723, 0.8666328191757202], [0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
7,5ifj_abc

In [97]:
df_2if = df_1if.drop(columns=["chai_unconstrained_model_path", "chai_constrained_model_path", "boltz_free_model_path", "boltz_template_model_path"])
df_2if

,sample,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq,af3_pdockq2,...,boltz_template_ipsae,boltz_template_pdockq,boltz_template_pdockq2,boltz_template_lis,binding,interface,chai_unconstrained_per_chain_ptm,chai_unconstrained_per_chain_pair_iptm,chai_constrained_per_chain_ptm,chai_constrained_per_chain_pair_iptm
0,5ifj_abc_cut,2.593883,95.404666,14.690,1.855,1.993750,95.036725,0.788184,0.20980,0.88085,...,0.937455,0.23935,0.96685,0.87115,False,"A,C","[[0.9473640322685242, 0.9504183530807495, 0.17...","[[[0.9473640322685242, 0.8909199237823486, 0.0...","[[0.9346736073493958, 0.941501259803772]]","[[[0.9346736073493958, 0.8659998178482056], [0..."
1,5ifj_abc_cut_A_D111A,2.815847,95.282854,13.225,1.745,2.100000,94.730775,0.765295,0.18660,0.88195,...,0.908519,0.21980,0.92900,0.85605,False,"A,C","[[0.9472975134849548, 0.9492976665496826, 0.17...","[[[0.9472975134849548, 0.8904723525047302, 0.0...","[[0.934756338596344, 0.9413727521896362]]","[[[0.934756338596344, 0.8664464950561523], [0...."
2,5ifj_abc_cut_A_K110A,2.790728,94.871984,19.975,2.845,2.275417,92.998114,0.769552,0.18030,0.85065,...,0.939275,0.23650,0.96220,0.87810,False,"A,C","[[0.9461949467658997, 0.9488812685012817, 0.16...","[[[0.9461949467658997, 0.8897757530212402, 0.0...","[[0.9350460767745972, 0.9415807127952576]]","[[[0.9350460767745972, 0.8660252094268799], [0..."
3,5ifj_abc_cut_A_S64A,2.539446,95.613237,14.595,1.810,2.072188,94.813927,0.778962,0.19900,0.88450,...,0.898855,0.22850,0.95750,0.84545,False,"A,C","[[0.9445436000823975, 0.9495869874954224, 0.17...","[[[0.9445436000823975, 0.8924098014831543, 0.0...","[[0.9345250129699707, 0.9413044452667236]]","[[[0.9345250129699707, 0.8659787178039551], [0..."
4,5ifj_abc_cut_A_W52A,3.005329,94.814333,46.435,5.940,2.303125,91.482646,0.695928,0.19125,0.78185,...,0.845670,0.21780,0.94175,0.81200,False,"A,C","[[0.9452440142631531, 0.9471760392189026, 0.16...","[[[0.9452440142631531, 0.8792759776115417, 0.0...","[[0.9345828890800476, 0.9411274194717407]]","[[[0.9345828890800476, 0.8663274645805359], [0..."
5,5ifj_abc_cut_A_Y66A,2.701376,95.337838,17.485,2.305,2.249649,93.429048,0.763715,0.19335,0.84245,...,0.892648,0.20680,0.95410,0.83435,False,"A,C","[[0.9473956227302551, 0.9488828778266907, 0.16...","[[[0.9473956227302551, 0.8850024938583374, 0.0...","[[0.9346519708633423, 0.9413310289382935]]","[[[0.9346519708633423, 0.8664906620979309], [0..."
6,5ifj_abc_cut_B_L116A,2.806607,95.032996,7.610,0.985,2.205952,93.484476,0.747441,0.17235,0.82990,...,0.905323,0.21785,0.94930,0.85275,False,"A,C","[[0.9485109448432922, 0.9499985575675964, 0.17...","[[[0.9485109448432922, 0.8920019865036011, 0.0...","[[0.9348855018615723, 0.9416057467460632]]","[[[0.9348855018615723, 0.8666328191757202], [0..."
7,5ifj_abc_cut_B_V38A,2.714935,95.220947,15.440,2.000,2.254062,93.465938,0.758796,0.19055,0.85940,...,0.840146,0.23650,0.94035,0.82200,False,"A,C","[[0.9468277096748352, 0.9472519159317017, 0.10...","[[[0.9468277096748352, 0.8879522085189819, 0.0...","[[0.9348711967468262, 0.9404247403144836]]","[[[0.9348711967468262, 0.8658405542373657], [0..."
8,5ifj_abc_cut_B_W107A,2.878966,95.089607,17.035,2.335,2.471875,93.380576,0.623665,0.16430,0.79175,...,0.851439,0.21350,0.93755,0.83195,False,"A,C","[[0.9425944089889526, 0.9466992616653442, 0.10...","[[[0.9425944089889526, 0.8834366798400879, 0.0...","[[0.9346067309379578, 0.9413726925849915]]","[[[0.9346067309379578, 0.8661914467811584], [0..."
9,5ifj_abc_cut_B_Y37A,3.004751,94.723352,10.755,1.380,2.298611,92.120377,0.640578,0.17745,0.77900,...,0.823940,0.19345,0.89155,0.81210,False,"A,C","[[0.940564751625061, 0.9448128938674927, 0.095...","[[[0.940564751625061, 0.8819113969802856, 0.04...","[[0.9345679879188538, 0.94112229347229]]","[[[0.9345679879188538, 0.8665298819541931], [0..."


## master df

##### snir


In [98]:
# for snir data
master_df = pd.read_csv(MASTER_PATH, delimiter=';')
master_df

,source,type,sample,fab,interface,vh/vl,target,target_residues,target_sequence,binder,KD[M],notes
0,snir_ab,original,5ifj_abc_cut,1000.0,"A,C",EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...,gliadin_alpha_E,NaN,PLQPEQPFP,1,0.000010,Fab 1E03 + KD[M] approximated from Snir et. al...
1,snir_ab,original,5ifj_abc_cut,1000.0,"B,C",QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...,gliadin_alpha_E,NaN,PLQPEQPFP,1,0.000010,Fab 1E03 + KD[M] approximated from Snir et. al...
2,snir_ab,original,5ig7_abc_cut,10.0,"A,C",EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...,gliadin_alpha_Q,NaN,PLQPQQPFP,1,0.000100,Fab 1E01 + KD[M] approximated from Snir et. al...
3,snir_ab,original,5ig7_abc_cut,10.0,"B,C",QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...,gliadin_alpha_Q,NaN,PLQPQQPFP,1,0.000100,Fab 1E01 + KD[M] approximated from Snir et. al...
4,snir_ab,original,5ijk_acx_cut,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,1,0.000001,Fab 1E01 + KD[M] approximated from Snir et. al...
...,...,...,...,...,...,...,...,...,...,...,...,...
65,snir_ab,alanine_scan,5ijk_acx_cut_B_Y108A,10.0,"B,C",DIVMTQSPDSLAVSLGERATINCKSSQSVLYSSNNKNYLAWYQQKP...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
66,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
67,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,10.0,"B,C",DIVMTQSPDSLAVSLGERATINCKSSQSVLASSNNKNYLAWYQQKP...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
68,snir_ab,alanine_scan,5ijk_acx_cut_B_Y38A,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN


In [99]:
master_df2 = master_df.drop(columns=["interface", "fab", "vh/vl", "target", "target_residues", "target_sequence", "notes"])
master_df2

,source,type,sample,binder,KD[M]
0,snir_ab,original,5ifj_abc_cut,1,0.000010
1,snir_ab,original,5ifj_abc_cut,1,0.000010
2,snir_ab,original,5ig7_abc_cut,1,0.000100
3,snir_ab,original,5ig7_abc_cut,1,0.000100
4,snir_ab,original,5ijk_acx_cut,1,0.000001
...,...,...,...,...,...
65,snir_ab,alanine_scan,5ijk_acx_cut_B_Y108A,0,NaN
66,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,0,NaN
67,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,0,NaN
68,snir_ab,alanine_scan,5ijk_acx_cut_B_Y38A,0,NaN


In [100]:
master_df3 = master_df2.drop_duplicates(keep="last")
master_df3

,source,type,sample,binder,KD[M]
1,snir_ab,original,5ifj_abc_cut,1,0.000010
3,snir_ab,original,5ig7_abc_cut,1,0.000100
5,snir_ab,original,5ijk_acx_cut,1,0.000001
7,snir_ab,alanine_scan,5ifj_abc_cut_A_D111A,0,NaN
9,snir_ab,alanine_scan,5ifj_abc_cut_A_K110A,0,NaN
11,snir_ab,alanine_scan,5ifj_abc_cut_A_S64A,0,NaN
13,snir_ab,alanine_scan,5ifj_abc_cut_A_W52A,0,NaN
15,snir_ab,alanine_scan,5ifj_abc_cut_A_Y66A,0,NaN
17,snir_ab,alanine_scan,5ifj_abc_cut_all_A,0,NaN
19,snir_ab,alanine_scan,5ifj_abc_cut_B_L116A,0,NaN


##### mcmahon / germinal

In [80]:
# for snir data
master_df = pd.read_csv(MASTER_PATH, delimiter=';')
master_df.head()

,source,type,vhh,vhh_sequence,target,target_residues,target_sequence,binder,KD[M],EC50[M],notes
0,mcmahon,original,Nb.b201,QVQLQESGGGLVQAGGSLRLSCAASGYISDAYYMGWYRQAPGKERE...,HSA,404-609,LVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVE...,1,4.310000e-07,NaN,NaN
1,mcmahon,original,Nb.b201,QVQLQESGGGLVQAGGSLRLSCAASGYISDAYYMGWYRQAPGKERE...,HSA,404-601,LVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVE...,1,4.310000e-07,NaN,NaN
2,mcmahon,original,Nb.b201,QVQLQESGGGLVQAGGSLRLSCAASGYISDAYYMGWYRQAPGKERE...,HSA,25-609,DAHKSEVAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEV...,1,4.310000e-07,NaN,NaN
3,mcmahon,original,Nb.AQ100,QVQLQESGGGLVQAGGSLRLSCAASGTIFRVSFMGWYRQAPGKERE...,Adiponectin,108-244,AYVYRSAFSVGLETYVTIPNMPIRFTKIFYNQQNHYDGSTGKFHCN...,1,2.300000e-06,NaN,NaN
4,mcmahon,original,Nb.AQ101,QVQLQESGGGLVQAGGSLRLSCAASGYIFVYAYMGWYRQAPGKERE...,Adiponectin,108-244,AYVYRSAFSVGLETYVTIPNMPIRFTKIFYNQQNHYDGSTGKFHCN...,0,NaN,NaN,NaN


In [81]:
def create_sample_id(row):
    vhh = str(row['vhh'])
    target = str(row['target'])
    residues = str(row['target_residues'])
    
    # Check if target is HSA to apply the specific naming convention
    if target == "HSA":
        return f"{vhh}_{target}_{residues}"
    else:
        return f"{vhh}_{target}"

# 1. Create the 'sample' column in the master table
master_df['sample'] = master_df.apply(create_sample_id, axis=1)
master_df



,source,type,vhh,vhh_sequence,target,target_residues,target_sequence,binder,KD[M],EC50[M],notes,sample
0,mcmahon,original,Nb.b201,QVQLQESGGGLVQAGGSLRLSCAASGYISDAYYMGWYRQAPGKERE...,HSA,404-609,LVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVE...,1,4.310000e-07,NaN,NaN,Nb.b201_HSA_404-609
1,mcmahon,original,Nb.b201,QVQLQESGGGLVQAGGSLRLSCAASGYISDAYYMGWYRQAPGKERE...,HSA,404-601,LVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVE...,1,4.310000e-07,NaN,NaN,Nb.b201_HSA_404-601
2,mcmahon,original,Nb.b201,QVQLQESGGGLVQAGGSLRLSCAASGYISDAYYMGWYRQAPGKERE...,HSA,25-609,DAHKSEVAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEV...,1,4.310000e-07,NaN,NaN,Nb.b201_HSA_25-609
3,mcmahon,original,Nb.AQ100,QVQLQESGGGLVQAGGSLRLSCAASGTIFRVSFMGWYRQAPGKERE...,Adiponectin,108-244,AYVYRSAFSVGLETYVTIPNMPIRFTKIFYNQQNHYDGSTGKFHCN...,1,2.300000e-06,NaN,NaN,Nb.AQ100_Adiponectin
4,mcmahon,original,Nb.AQ101,QVQLQESGGGLVQAGGSLRLSCAASGYIFVYAYMGWYRQAPGKERE...,Adiponectin,108-244,AYVYRSAFSVGLETYVTIPNMPIRFTKIFYNQQNHYDGSTGKFHCN...,0,NaN,NaN,NaN,Nb.AQ101_Adiponectin
5,mcmahon,original,Nb.AQ102,QVQLQESGGGLVQAGGSLRLSCAASGSIFRVAWMGWYRQAPGKERE...,Adiponectin,108-244,AYVYRSAFSVGLETYVTIPNMPIRFTKIFYNQQNHYDGSTGKFHCN...,0,NaN,NaN,NaN,Nb.AQ102_Adiponectin
6,mcmahon,original,Nb.AQ103,QVQLQESGGGLVQAGGSLRLSCAASGSIFAAAYMGWYRQAPGKERE...,Adiponectin,108-244,AYVYRSAFSVGLETYVTIPNMPIRFTKIFYNQQNHYDGSTGKFHCN...,1,5.100000e-07,NaN,NaN,Nb.AQ103_Adiponectin
7,mcmahon,original,Nb.AQ104,QVQLQESGGGLVQAGGSLRLSCAASGYIFPSAVMGWYRQAPGKERE...,Adiponectin,108-244,AYVYRSAFSVGLETYVTIPNMPIRFTKIFYNQQNHYDGSTGKFHCN...,0,NaN,NaN,NaN,Nb.AQ104_Adiponectin
8,mcmahon,original,Nb.AD101,QVQLQESGGGLVQAGGSLRLSCAASGTISPLPAMGWYRQAPGKERE...,A2AR,6-309,SSVYITVELAIAVLAILGNVLVCWAVWLNSNLQNVTNYFVVSLAAA...,0,NaN,NaN,NaN,Nb.AD101_A2AR
9,mcmahon,original,Nb.AD102,QVQLQESGGGLVQAGGSLRLSCAASGSIFSILSMGWYRQAPGKERE...,A2AR,6-309,SSVYITVELAIAVLAILGNVLVCWAVWLNSNLQNVTNYFVVSLAAA...,0,NaN,NaN,May have reduced binding on freeze/thaw,Nb.AD102_A2AR


In [82]:
master_df2 = master_df.drop(columns=["vhh", "target", "target_residues", "target_sequence", "notes"])
master_df2

,source,type,vhh_sequence,binder,KD[M],EC50[M],sample
0,mcmahon,original,QVQLQESGGGLVQAGGSLRLSCAASGYISDAYYMGWYRQAPGKERE...,1,4.310000e-07,NaN,Nb.b201_HSA_404-609
1,mcmahon,original,QVQLQESGGGLVQAGGSLRLSCAASGYISDAYYMGWYRQAPGKERE...,1,4.310000e-07,NaN,Nb.b201_HSA_404-601
2,mcmahon,original,QVQLQESGGGLVQAGGSLRLSCAASGYISDAYYMGWYRQAPGKERE...,1,4.310000e-07,NaN,Nb.b201_HSA_25-609
3,mcmahon,original,QVQLQESGGGLVQAGGSLRLSCAASGTIFRVSFMGWYRQAPGKERE...,1,2.300000e-06,NaN,Nb.AQ100_Adiponectin
4,mcmahon,original,QVQLQESGGGLVQAGGSLRLSCAASGYIFVYAYMGWYRQAPGKERE...,0,NaN,NaN,Nb.AQ101_Adiponectin
5,mcmahon,original,QVQLQESGGGLVQAGGSLRLSCAASGSIFRVAWMGWYRQAPGKERE...,0,NaN,NaN,Nb.AQ102_Adiponectin
6,mcmahon,original,QVQLQESGGGLVQAGGSLRLSCAASGSIFAAAYMGWYRQAPGKERE...,1,5.100000e-07,NaN,Nb.AQ103_Adiponectin
7,mcmahon,original,QVQLQESGGGLVQAGGSLRLSCAASGYIFPSAVMGWYRQAPGKERE...,0,NaN,NaN,Nb.AQ104_Adiponectin
8,mcmahon,original,QVQLQESGGGLVQAGGSLRLSCAASGTISPLPAMGWYRQAPGKERE...,0,NaN,NaN,Nb.AD101_A2AR
9,mcmahon,original,QVQLQESGGGLVQAGGSLRLSCAASGSIFSILSMGWYRQAPGKERE...,0,NaN,NaN,Nb.AD102_A2AR


### merge

In [101]:
df = pd.merge(master_df3, df_2if, on=['sample'], how='inner')
#df = pd.merge(master_df2, preds_df, on=['sample'], how='inner')
# Display the first few rows to verify
print(df[['sample', 'af3_plddt']])

# Save the merged result
df.to_csv(OUTPUT_DIR /'merged_snir.csv', index=False)

                  sample  af3_plddt
0           5ifj_abc_cut  95.404666
1           5ig7_abc_cut  95.599717
2           5ijk_acx_cut  92.439163
3   5ifj_abc_cut_A_D111A  95.282854
4   5ifj_abc_cut_A_K110A  94.871984
5    5ifj_abc_cut_A_S64A  95.613237
6    5ifj_abc_cut_A_W52A  94.814333
7    5ifj_abc_cut_A_Y66A  95.337838
8     5ifj_abc_cut_all_A  93.235998
9   5ifj_abc_cut_B_L116A  95.032996
10   5ifj_abc_cut_B_V38A  95.220947
11  5ifj_abc_cut_B_W107A  95.089607
12   5ifj_abc_cut_B_Y37A  94.723352
13  5ig7_abc_cut_A_K110A  94.950463
14   5ig7_abc_cut_A_W52A  95.457781
15   5ig7_abc_cut_A_Y66A  95.767986
16    5ig7_abc_cut_all_A  92.210506
17  5ig7_abc_cut_B_G114A  95.869087
18  5ig7_abc_cut_B_L116A  95.298420
19   5ig7_abc_cut_B_V38A  95.793462
20  5ig7_abc_cut_B_W107A  92.830579
21   5ig7_abc_cut_B_Y37A  95.631059
22  5ijk_acx_cut_A_I112A  94.094014
23  5ijk_acx_cut_A_I114A  91.142842
24   5ijk_acx_cut_A_K57A  92.231214
25   5ijk_acx_cut_A_R55A  91.770396
26  5ijk_acx_cut_A_T113A  97